In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib

In [61]:
def data_load():
    df = pd.read_csv("data/Housing.csv")
    return df

In [62]:
def preprocess(df):
    qua25 = df["price"].quantile(0.25)
    qua75 = df["price"].quantile(0.75)
    
    iqr = qua75 - qua25
    
    upper_limit = qua75 + 1.5 * iqr
    lower_limit = qua25 - 1.5 * iqr

    df['price_per_area'] = df['price'] / df['area']
    df['total_rooms'] = df['bedrooms'] + df['bathrooms']
    df['area_per_room'] = df['area'] / df['bedrooms']
    df['price'] = np.where(
        df['price'] > upper_limit,
        upper_limit,
        np.where(
            df['price'] < lower_limit,
            lower_limit,
            df['price']
        )
    )
    return df

In [81]:
def train_model(df):
    categorial_cols = ["mainroad","guestroom","basement","hotwaterheating","airconditioning","prefarea","furnishingstatus"]
    numeric_cols = ["area","bedrooms","bathrooms","stories","parking","price_per_area","total_rooms","area_per_room"]
    transformers = ColumnTransformer(
    transformers=[
            ("categorial", OneHotEncoder(drop="first", sparse_output=True), categorial_cols),
            ("numeric", StandardScaler(), numeric_cols)
        ]
    )
    lr = LinearRegression()
    pipe = Pipeline(
    steps=[
        ("preproceeser", transformers),
        ("classifier", lr)
        ]
    )

    x = df.drop("price", axis=1)
    y = df["price"]
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=42)
    
    
    pipe.fit(x_train, y_train)
    joblib.dump(pipe, "house_price_model.pkl")
    y_pred = pipe.predict(x_test)
    evalute(y_test, y_pred)
    

In [82]:
def evalute(y_test, y_pred):
    print("MSE : ", mean_squared_error(y_test, y_pred))
    print("MAE : ", mean_absolute_error(y_test, y_pred))
    print("R2 Score : ", r2_score(y_test, y_pred))

In [83]:
def main():
    df = data_load()
    df = preprocess(df)
    train_model(df)

In [84]:
if __name__ == "__main__":
    main()

MSE :  238368265864.10452
MAE :  382044.179720216
R2 Score :  0.9319308831168386
